# ICS Core - Sensor Validation Notebook

Combined notebook covering the five-sensor downhole package for the dsPIC33AK256MC205-H/M7 firmware. Models first, then pytest test suites, then `generate_dashboard` runs the tests and produces the validation HTML.

| Section | Content |
|---|---|
| 0 | Workspace setup (creates sandbox directory) |
| 1 | Sensor models written directly to the sandbox |
| 2 | pytest test suites written directly to the sandbox |
| 3 | `generate_dashboard()` function |
| 4 | Run dashboard, return HTML file path |

**Sandbox layout:** all model and test code is written to `./ics_sandbox/` using `%%writefile`. Cells contain plain readable Python - no encoding, no string wrapping. Pytest runs against the sandbox; the dashboard HTML is written to `./ics_sandbox/reports/sensor_dashboard.html`.


---
## Section 0 - Workspace Setup

Creates the sandbox directory tree and the empty `__init__.py` files pytest needs to recognise the package layout. Must run before any `%%writefile` cell below.


In [ ]:
import os, sys, json, subprocess
from pathlib import Path

# Sandbox root - every %%writefile cell below uses this exact path
SANDBOX = Path("ics_sandbox").resolve()

# Directory tree mirrors the original simulation/ + tests/ layout
SIM_MODELS_DIR = SANDBOX / "simulation" / "models"
TESTS_DIR      = SANDBOX / "tests"
REPORTS_DIR    = SANDBOX / "reports"

for d in [SIM_MODELS_DIR, TESTS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Empty __init__.py so pytest can import the package
(SANDBOX / "simulation" / "__init__.py").write_text("")
(SIM_MODELS_DIR / "__init__.py").write_text("")
(TESTS_DIR / "__init__.py").write_text("")

print(f"Sandbox ready: {SANDBOX}")


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "pytest-json-report"], check=True)

---
## Section 1 - Sensor Models

Each cell writes one model file into the sandbox via `%%writefile`. Cell content is the actual model code - readable as plain Python.


### 1.1 - ADXL206HDZ Dual-Axis Accelerometer
Sensitivity 328 mV/g, range +-5 g, max temp 175 C. Used for inclination measurement.

In [ ]:
%%writefile ics_sandbox/simulation/models/adxl206_model.py
"""
ADXL206HDZ Dual-Axis MEMS Accelerometer Simulation
Sensitivity: 328 mV/g  |  Range: ±5 g  |  VCC: 5.0 V  |  Max Temp: 175°C
Output: XOUT, YOUT ratiometric to VCC
"""
import numpy as np
from scipy.signal import butter, sosfilt

VCC                = 5.0    # Supply voltage (V)
SENSITIVITY_MV_G   = 328.0  # mV/g at 5V (datasheet typical)
SENSITIVITY_V_G    = SENSITIVITY_MV_G / 1000.0
ZERO_G_OUTPUT_V    = VCC / 2.0   # 2.5 V at 0 g
RANGE_G            = 5.0
RESONANT_FREQ_HZ   = 5500.0  # Typical MEMS resonance for bandwidth set by Cx/Cy
DAMPING_RATIO      = 0.70    # Critical damping approximation

# Temperature coefficient of sensitivity: −0.01%/°C (datasheet)
TEMP_COEFF_SENS    = -0.0001
NOISE_FLOOR_G_RMS  = 0.001   # 1 mg noise floor (110 µg/√Hz × √(BW))


def sensitivity_at_temp(temp_c):
    """Return adjusted sensitivity (V/g) at a given temperature."""
    delta = 1.0 + TEMP_COEFF_SENS * (temp_c - 25.0)
    return SENSITIVITY_V_G * delta

def acceleration_to_voltage(accel_g, temp_c=25.0, noise=True):
    """
    Convert acceleration (g) to ADXL206 output voltage.
    Clamps to ±5 g. Adds noise floor if requested.
    Returns: voltage (V)
    """
    accel_g = np.clip(accel_g, -RANGE_G, RANGE_G)
    sens = sensitivity_at_temp(temp_c)
    v_out = ZERO_G_OUTPUT_V + accel_g * sens
    if noise:
        v_out += np.random.normal(0, NOISE_FLOOR_G_RMS * sens,
                                   np.asarray(accel_g).shape or (1,))
    return v_out

def inclination_from_dual_axis(x_g, y_g):
    """
    Compute inclination angle (degrees) from dual-axis reading.
    Uses arctangent. Returns (pitch, roll) in degrees.
    """
    pitch_rad = np.arctan2(x_g, np.sqrt(y_g**2 + 1.0))
    roll_rad  = np.arctan2(y_g, np.sqrt(x_g**2 + 1.0))
    return np.degrees(pitch_rad), np.degrees(roll_rad)

def simulate_static_tilt(pitch_deg, roll_deg, temp_c=25.0):
    """
    Given a known tilt, return simulated XOUT/YOUT voltages.
    Converts angles to g-components via gravity projection.
    """
    x_g = np.sin(np.radians(pitch_deg))
    y_g = np.sin(np.radians(roll_deg))
    vx = acceleration_to_voltage(x_g, temp_c, noise=False)
    vy = acceleration_to_voltage(y_g, temp_c, noise=False)
    return vx, vy


### 1.2 - ADXRS645HDYZ Rate Gyroscope
Sensitivity 12.5 mV per dps, range +-2000 dps. RPM and rotation tracking.

In [ ]:
%%writefile ics_sandbox/simulation/models/adxrs645_model.py
"""
ADXRS645HDYZ MEMS Rate Gyroscope Simulation
Sensitivity: ~12.5 mV/(°/s)  |  Range: ±2000°/s  |  VRATIO: 5.0 V
Max Temp: 175°C
"""
import numpy as np

VRATIO             = 5.0
SENSITIVITY_MV_DPS = 12.5           # mV per degree per second
SENSITIVITY_V_DPS  = SENSITIVITY_MV_DPS / 1000.0
NULL_RATE_V        = VRATIO / 2.0   # 2.5 V at 0 °/s
RANGE_DPS          = 2000.0
NOISE_DPS_RMS      = 0.5            # Typical noise floor
BIAS_DPS           = 0.0            # Can be set to simulate bias error

# Temperature coefficient of null: 0.015 °/s/°C
TEMP_COEFF_NULL    = 0.015


def rate_to_voltage(rate_dps, temp_c=25.0, noise=True):
    """
    Convert angular rate (°/s) to ADXRS645 RATEOUT voltage.
    Includes bias, temperature-dependent null shift, and noise.
    """
    rate_dps = np.clip(rate_dps, -RANGE_DPS, RANGE_DPS)
    null_shift = TEMP_COEFF_NULL * (temp_c - 25.0)
    effective_rate = rate_dps + BIAS_DPS + null_shift
    v_out = NULL_RATE_V + effective_rate * SENSITIVITY_V_DPS
    if noise:
        n = np.random.normal(0, NOISE_DPS_RMS * SENSITIVITY_V_DPS,
                              np.asarray(rate_dps).shape or (1,))
        v_out += n
    return v_out

def voltage_to_rate(v_out, temp_c=25.0):
    """Reverse: voltage → angular rate (°/s), with temperature null correction."""
    null_shift = TEMP_COEFF_NULL * (temp_c - 25.0)
    return (v_out - NULL_RATE_V) / SENSITIVITY_V_DPS - null_shift

def integrate_rate_to_angle(rate_dps_array, dt_s):
    """Integrate angular rate to give angle (degrees). Simple Euler integration."""
    return np.cumsum(rate_dps_array) * dt_s


### 1.3 - AT/10/TB IEPE Accelerometer Chain
10 mV/g, +-500 g range. Full signal chain: ICP -> AC couple -> 1.65 V offset -> 4.8 kHz AA filter -> 12-bit ADC.

In [ ]:
%%writefile ics_sandbox/simulation/models/at10tb_model.py
"""
AT/10/TB IEPE Accelerometer Signal Chain Simulation
Sensitivity: 10 mV/g  |  Range: ±500 g  |  Max Temp: 165°C
Signal chain: ICP source → AC couple → 1.65V offset → 4.8kHz AA filter → ADC
"""
import numpy as np
from scipy.signal import butter, sosfilt

SENSITIVITY_V_PER_G = 0.010   # 10 mV/g
RANGE_G             = 500.0   # ±500 g
DC_OFFSET_V         = 1.65    # Summed bias to centre signal in 0–3.3 V window
ADC_REF_V           = 3.3     # dsPIC33AK ADC reference
ADC_BITS            = 12      # 12-bit ADC
AA_CUTOFF_HZ        = 4800.0  # Anti-alias filter cutoff
SAMPLE_RATE_HZ      = 50000   # 50 kSa/s

def butter_lowpass(cutoff, fs, order=4):
    nyq = fs / 2.0
    normal_cutoff = cutoff / nyq
    return butter(order, normal_cutoff, btype='low', analog=False, output='sos')

def generate_vibration(freq_hz=1000.0, amplitude_g=10.0, duration_s=0.01,
                        noise_g_rms=0.05, fs=SAMPLE_RATE_HZ):
    """Generate synthetic vibration signal in g-units."""
    t = np.arange(0, duration_s, 1.0/fs)
    signal_g = amplitude_g * np.sin(2 * np.pi * freq_hz * t)
    noise    = np.random.normal(0, noise_g_rms, len(t))
    return t, signal_g + noise

def apply_signal_chain(signal_g, fs=SAMPLE_RATE_HZ):
    """
    Apply full AT/10/TB signal chain.
    Returns voltage array at ADC input (0 to 3.3 V).
    """
    # Step 1: Convert g to voltage (sensitivity)
    v_ac = signal_g * SENSITIVITY_V_PER_G

    # Step 2: AC coupling — remove any DC (tantalum cap model: high-pass at ~0.016 Hz)
    # For simulation we assume perfect DC rejection; signal is already AC
    v_ac_coupled = v_ac

    # Step 3: Add 1.65 V DC offset (OPA333AQ summing stage)
    v_offset = v_ac_coupled + DC_OFFSET_V

    # Step 4: Anti-alias Butterworth filter (4.8 kHz, 4th order)
    sos = butter_lowpass(AA_CUTOFF_HZ, fs, order=4)
    v_filtered = sosfilt(sos, v_offset)

    # Step 5: Clamp to ADC rail (0–3.3 V)
    v_clamped = np.clip(v_filtered, 0.0, ADC_REF_V)
    return v_clamped

def voltage_to_counts(v_array):
    """Convert voltage array to 12-bit ADC counts."""
    counts = (v_array / ADC_REF_V) * (2**ADC_BITS - 1)
    return np.round(counts).astype(int)

def counts_to_g(counts):
    """Reverse: ADC counts → g-units (for firmware validation)."""
    v = (counts / (2**ADC_BITS - 1)) * ADC_REF_V
    return (v - DC_OFFSET_V) / SENSITIVITY_V_PER_G


### 1.4 - LM95172 SPI Temperature Sensor
-40 to 200 C, 13-16 bit configurable. Full register map mock for SPI driver testing.

In [ ]:
%%writefile ics_sandbox/simulation/models/lm95172_mock.py
"""
LM95172 SPI Digital Temperature Sensor Mock
Range: −40°C to 200°C  |  Resolution: 13–16 bit  |  Interface: SPI 3-wire
Simulates full register map for firmware SPI driver testing.
"""
import struct

# Register addresses
REG_TEMP   = 0x00   # Temperature register (read)
REG_CONFIG = 0x01   # Configuration register (read/write)

# Configuration bits
CFG_RESOLUTION_13 = 0b00  # 13-bit: 0.0625°C/LSB
CFG_RESOLUTION_14 = 0b01
CFG_RESOLUTION_15 = 0b10
CFG_RESOLUTION_16 = 0b11  # 16-bit: 0.0078125°C/LSB

RESOLUTION_LSB = {
    CFG_RESOLUTION_13: 0.0625,
    CFG_RESOLUTION_14: 0.03125,
    CFG_RESOLUTION_15: 0.015625,
    CFG_RESOLUTION_16: 0.0078125,
}

TEMP_MIN_C = -40.0
TEMP_MAX_C = 200.0
ACCURACY_C = 1.0    # ±1°C (datasheet)


class LM95172Mock:
    """
    Software mock of LM95172 temperature sensor.
    Inject temperature via set_temperature().
    Call spi_read() to get formatted 16-bit SPI word as firmware would see it.
    """
    def __init__(self, resolution=CFG_RESOLUTION_13):
        self._temp_c    = 25.0
        self._config    = resolution
        self._fault     = False

    def set_temperature(self, temp_c):
        """Inject a known temperature for test validation."""
        if not (TEMP_MIN_C <= temp_c <= TEMP_MAX_C):
            raise ValueError(f"Temperature {temp_c}°C out of sensor range")
        self._temp_c = temp_c

    def spi_read_temperature(self):
        """
        Return 16-bit SPI word as the LM95172 would output.
        Bits [15:3] = temperature (two's complement, left-aligned at 13-bit).
        Bit [2]     = conversion complete flag (1 = ready).
        """
        lsb = RESOLUTION_LSB[self._config]
        raw_int = int(round(self._temp_c / lsb))
        # Clamp to 13-bit signed range
        raw_int = max(-4096, min(4095, raw_int))
        # Left-align in 16-bit word (shift left by 3 for 13-bit mode)
        shift = {0b00: 3, 0b01: 2, 0b10: 1, 0b11: 0}[self._config]
        word = (raw_int << shift) & 0xFFFF
        word |= 0x0004  # Conversion complete bit
        return word

    def parse_temperature(self, spi_word):
        """
        Parse a 16-bit SPI word back to temperature (°C).
        Mirrors what firmware would do after reading from the sensor.
        """
        shift = {0b00: 3, 0b01: 2, 0b10: 1, 0b11: 0}[self._config]
        raw = (spi_word >> shift) & 0x1FFF  # 13 bits
        if raw & 0x1000:  # Sign bit
            raw -= 0x2000
        lsb = RESOLUTION_LSB[self._config]
        return raw * lsb


### 1.5 - MAX31856 Thermocouple-to-Digital Converter
Type K, -270 to 1800 C. Full register map including fault injection.

In [ ]:
%%writefile ics_sandbox/simulation/models/max31856_mock.py
"""
MAX31856 Thermocouple-to-Digital Converter Mock (Type K)
TC Range: −270°C to 1800°C  |  CJC Range: −40°C to 125°C
Resolution: 0.0078125°C (19-bit)  |  Interface: SPI 4-wire
"""
import numpy as np

# Register map (addresses match MAX31856 datasheet Table 1)
REG_CR0   = 0x00  # Config register 0
REG_CR1   = 0x01  # Config register 1 (thermocouple type)
REG_MASK  = 0x02  # Fault mask
REG_CJHF  = 0x03  # CJ high fault threshold
REG_CJLF  = 0x04  # CJ low fault threshold
REG_LTHFTH= 0x05  # LTC high fault MSB
REG_LTHFTL= 0x06  # LTC high fault LSB
REG_LTLFTH= 0x07  # LTC low fault MSB
REG_LTLFTL= 0x08  # LTC low fault LSB
REG_CJTO  = 0x09  # CJ temperature offset
REG_CJTH  = 0x0A  # CJ temperature MSB
REG_CJTL  = 0x0B  # CJ temperature LSB
REG_LTCBH = 0x0C  # Linearised TC temperature MSB
REG_LTCBM = 0x0D  # Linearised TC temperature mid
REG_LTCBL = 0x0E  # Linearised TC temperature LSB
REG_SR    = 0x0F  # Fault status register

# Fault bits (SR register)
FAULT_CJ_RANGE  = 0x80
FAULT_TC_RANGE  = 0x40
FAULT_CJ_HIGH   = 0x20
FAULT_CJ_LOW    = 0x10
FAULT_TC_HIGH   = 0x08
FAULT_TC_LOW    = 0x04
FAULT_OV_UV     = 0x02
FAULT_OPEN      = 0x01

TC_RESOLUTION   = 0.0078125  # °C per LSB (19-bit)
CJ_RESOLUTION   = 0.015625   # °C per LSB (14-bit)


def seebeck_voltage_uv(temp_c):
    """
    Approximate Type K Seebeck voltage (µV) relative to 0°C reference.
    Uses piecewise polynomial approximation (ITS-90 data).
    """
    t = temp_c
    if t < 0:
        return (38.74052e0 * t + 3.329941e-2 * t**2 + 2.061824e-4 * t**3 +
                -2.188225e-6 * t**4)
    else:
        return (39.45013e0 * t - 4.371092e-3 * t**2 + 1.517342e-5 * t**3 +
                -1.028760e-7 * t**4)


class MAX31856Mock:
    """
    Full register-map simulation of MAX31856.
    Inject tc_temp_c and cj_temp_c, call spi_read_register() to get formatted bytes
    exactly as the real chip would return over SPI.
    """
    def __init__(self):
        self._tc_temp_c   = 25.0    # Injected thermocouple temperature
        self._cj_temp_c   = 25.0    # Injected cold-junction temperature
        self._fault_inject = 0x00   # Inject fault bits for fault-detection testing
        self._registers   = {
            REG_CR0: 0x00, REG_CR1: 0x03,  # CR1: Type K default
            REG_MASK: 0xFF, REG_SR: 0x00,
        }

    def set_tc_temperature(self, temp_c):
        """Inject thermocouple (external) temperature."""
        self._tc_temp_c = temp_c

    def set_cj_temperature(self, temp_c):
        """Inject cold-junction (board-level) temperature."""
        if not (-40.0 <= temp_c <= 125.0):
            raise ValueError(f"CJ temp {temp_c}°C out of MAX31856 CJ range")
        self._cj_temp_c = temp_c

    def inject_fault(self, fault_bits):
        """Set fault bits to test fault detection logic in firmware."""
        self._fault_inject = fault_bits

    def _tc_raw(self):
        """Return 19-bit signed integer for TC temperature register."""
        raw = int(round(self._tc_temp_c / TC_RESOLUTION))
        return max(-1 * 2**18, min(2**18 - 1, raw))

    def _cj_raw(self):
        """Return 14-bit signed integer for CJ temperature register."""
        raw = int(round(self._cj_temp_c / CJ_RESOLUTION))
        return max(-1 * 2**13, min(2**13 - 1, raw))

    def spi_read_register(self, reg_addr):
        """
        Return the byte(s) the MAX31856 outputs for a given register read.
        Multi-byte registers return list of bytes (MSB first).
        """
        if reg_addr == REG_CJTH:
            raw = self._cj_raw()
            msb = (raw >> 6) & 0xFF
            lsb = (raw & 0x3F) << 2
            return [msb, lsb]

        elif reg_addr == REG_LTCBH:
            raw = self._tc_raw()
            # 19-bit: bits [18:11] in byte0, [10:3] in byte1, [2:0] in byte2 top bits
            b0 = (raw >> 11) & 0xFF
            b1 = (raw >> 3)  & 0xFF
            b2 = (raw & 0x07) << 5
            return [b0, b1, b2]

        elif reg_addr == REG_SR:
            return [self._fault_inject]

        return [self._registers.get(reg_addr, 0x00)]

    def parse_tc_temperature(self, b0, b1, b2):
        """Parse three LTCB bytes back to temperature (°C)."""
        raw = ((b0 << 11) | (b1 << 3) | (b2 >> 5))
        if raw & (1 << 18):  # Sign bit
            raw -= (1 << 19)
        return raw * TC_RESOLUTION

    def parse_cj_temperature(self, msb, lsb):
        """Parse CJ register bytes to temperature (°C)."""
        raw = ((msb << 6) | (lsb >> 2))
        if raw & (1 << 13):  # Sign bit
            raw -= (1 << 14)
        return raw * CJ_RESOLUTION


---
## Section 2 - pytest Test Suites

Each cell writes one test file into the sandbox via `%%writefile`. Tests import their target models from `simulation.models.<name>` exactly as in the original project layout.


### 2.1 - ADXL206HDZ Tests
Zero-g output, scale factor, range clamping, temperature drift, dual-axis inclination math.

In [ ]:
%%writefile ics_sandbox/tests/test_adxl206.py
import pytest
import numpy as np
from simulation.models.adxl206_model import (
    acceleration_to_voltage,
    inclination_from_dual_axis,
    simulate_static_tilt,
    ZERO_G_OUTPUT_V,
    SENSITIVITY_V_G,
    VCC,
)

V_TOL = 0.015
ANG_TOL = 1.0


class TestADXL206:

    def test_zero_g_output(self):
        v = acceleration_to_voltage(0.0, noise=False)
        assert abs(v - ZERO_G_OUTPUT_V) < V_TOL

    def test_positive_1g_output(self):
        expected = ZERO_G_OUTPUT_V + 1.0 * SENSITIVITY_V_G
        v = acceleration_to_voltage(1.0, noise=False)
        assert abs(v - expected) < V_TOL

    def test_negative_1g_output(self):
        expected = ZERO_G_OUTPUT_V - 1.0 * SENSITIVITY_V_G
        v = acceleration_to_voltage(-1.0, noise=False)
        assert abs(v - expected) < V_TOL

    def test_full_scale_positive(self):
        v = acceleration_to_voltage(5.0, noise=False)
        assert v <= VCC and v >= 0.0

    def test_over_range_clamps(self):
        v_over = acceleration_to_voltage(10.0, noise=False)
        v_max = acceleration_to_voltage(5.0, noise=False)
        assert abs(v_over - v_max) < V_TOL

    def test_temperature_sensitivity_shift(self):
        from simulation.models.adxl206_model import sensitivity_at_temp, TEMP_COEFF_SENS
        s25 = sensitivity_at_temp(25.0)
        s175 = sensitivity_at_temp(175.0)
        expected_ratio = 1.0 + TEMP_COEFF_SENS * (175.0 - 25.0)
        assert abs(s175 / s25 - expected_ratio) < 0.005

    def test_inclination_45_degrees(self):
        vx, vy = simulate_static_tilt(pitch_deg=45.0, roll_deg=0.0)
        x_g = (vx - ZERO_G_OUTPUT_V) / SENSITIVITY_V_G
        y_g = (vy - ZERO_G_OUTPUT_V) / SENSITIVITY_V_G
        pitch, _ = inclination_from_dual_axis(x_g, y_g)
        assert abs(pitch - 45.0) < ANG_TOL

    def test_level_gives_zero_inclination(self):
        vx, vy = simulate_static_tilt(0.0, 0.0)
        x_g = (vx - ZERO_G_OUTPUT_V) / SENSITIVITY_V_G
        y_g = (vy - ZERO_G_OUTPUT_V) / SENSITIVITY_V_G
        pitch, roll = inclination_from_dual_axis(x_g, y_g)
        assert abs(pitch) < ANG_TOL and abs(roll) < ANG_TOL


### 2.2 - ADXRS645HDYZ Tests
Null voltage, sensitivity, voltage<->rate round-trip, integration accuracy, temperature null shift.

In [ ]:
%%writefile ics_sandbox/tests/test_adxrs645.py
"""pytest tests for ADXRS645 gyroscope simulation"""
import numpy as np
from simulation.models.adxrs645_model import (
    rate_to_voltage, voltage_to_rate,
    integrate_rate_to_angle, NULL_RATE_V, SENSITIVITY_V_DPS
)

V_TOL    = 0.050  # ±50 mV
RATE_TOL = 5.0    # ±5 °/s round-trip tolerance
ANGLE_TOL = 2.0   # ±2° integration tolerance over 1 second

class TestADXRS645:

    def test_zero_rate_midpoint(self):
        """At 0 °/s, RATEOUT must be VRATIO/2 = 2.5 V"""
        v = rate_to_voltage(0.0, noise=False)
        assert abs(v - NULL_RATE_V) < V_TOL

    def test_positive_rate_voltage(self):
        """At +100 °/s, output = 2.5 + 100 * 0.0125 = 3.75 V"""
        expected = NULL_RATE_V + 100.0 * SENSITIVITY_V_DPS
        v = rate_to_voltage(100.0, noise=False)
        assert abs(v - expected) < V_TOL

    def test_negative_rate_voltage(self):
        """At −100 °/s, output = 2.5 − 1.25 = 1.25 V"""
        expected = NULL_RATE_V - 100.0 * SENSITIVITY_V_DPS
        v = rate_to_voltage(-100.0, noise=False)
        assert abs(v - expected) < V_TOL

    def test_round_trip_rate_accuracy(self):
        """voltage_to_rate(rate_to_voltage(r)) ≈ r"""
        rates = np.array([-2000, -500, -100, 0, 100, 500, 2000], dtype=float)
        for r in rates:
            v = rate_to_voltage(r, noise=False)
            recovered = voltage_to_rate(v)
            assert abs(recovered - r) < RATE_TOL, f"Rate {r} °/s failed round-trip"

    def test_integrate_constant_rate(self):
        """Integrating 90 °/s for 1 second must yield ~90 degrees."""
        dt = 0.001  # 1 ms sample interval
        N  = int(1.0 / dt)
        rates = np.full(N, 90.0)
        angles = integrate_rate_to_angle(rates, dt)
        assert abs(angles[-1] - 90.0) < ANGLE_TOL

    def test_temperature_null_shift(self):
        """At 175°C, null shift must move 0 °/s reading by ~2.25 V equivalent"""
        v_25  = rate_to_voltage(0.0, temp_c=25.0,  noise=False)
        v_175 = rate_to_voltage(0.0, temp_c=175.0, noise=False)
        from simulation.models.adxrs645_model import TEMP_COEFF_NULL
        expected_shift_v = TEMP_COEFF_NULL * (175.0 - 25.0) * SENSITIVITY_V_DPS
        assert abs((v_175 - v_25) - expected_shift_v) < V_TOL


### 2.3 - AT/10/TB Signal Chain Tests
DC offset, peak voltages, range non-clipping, ADC round-trip, anti-alias filter rejection.

In [ ]:
%%writefile ics_sandbox/tests/test_at10tb.py
"""pytest test suite for AT/10/TB simulation model"""
import pytest
import numpy as np
from simulation.models.at10tb_model import (
    generate_vibration, apply_signal_chain,
    voltage_to_counts, counts_to_g,
    SENSITIVITY_V_PER_G, DC_OFFSET_V, ADC_REF_V
)

TOLERANCE_G     = 2.0   # ±2 g round-trip tolerance
VOLTAGE_TOL_V   = 0.05  # ±50 mV voltage tolerance

class TestAT10TBSignalChain:

    def test_zero_g_gives_midpoint_voltage(self):
        """At 0 g input the ADC voltage must equal DC_OFFSET_V (1.65 V)"""
        signal_g = np.zeros(1000)
        v = apply_signal_chain(signal_g)
        assert abs(np.mean(v) - DC_OFFSET_V) < VOLTAGE_TOL_V, (
            f"Expected ~{DC_OFFSET_V} V at 0 g, got {np.mean(v):.4f} V")

    def test_positive_peak_voltage(self):
        """100 g peak → voltage peak near 1.65 + (100*0.010) = 2.65 V"""
        _, sig = generate_vibration(freq_hz=100, amplitude_g=100, noise_g_rms=0)
        v = apply_signal_chain(sig)
        expected_peak = DC_OFFSET_V + 100 * SENSITIVITY_V_PER_G
        assert abs(np.max(v) - expected_peak) < VOLTAGE_TOL_V, (
            f"Expected peak ~{expected_peak:.3f} V, got {np.max(v):.4f} V")

    def test_negative_peak_voltage(self):
        """−100 g peak → voltage trough near 1.65 − 1.0 = 0.65 V"""
        _, sig = generate_vibration(freq_hz=100, amplitude_g=100, noise_g_rms=0)
        v = apply_signal_chain(sig)
        expected_trough = DC_OFFSET_V - 100 * SENSITIVITY_V_PER_G
        assert abs(np.min(v) - expected_trough) < VOLTAGE_TOL_V

    def test_max_range_does_not_clip(self):
        """±500 g must NOT saturate the 0–3.3 V ADC rail"""
        _, sig = generate_vibration(freq_hz=100, amplitude_g=500, noise_g_rms=0)
        v = apply_signal_chain(sig)
        assert np.max(v) <= ADC_REF_V + 0.001
        assert np.min(v) >= -0.001

    def test_round_trip_g_accuracy(self):
        """counts_to_g(voltage_to_counts(apply_signal_chain(sig))) ≈ input g"""
        amp_g = 50.0
        _, sig = generate_vibration(freq_hz=200, amplitude_g=amp_g, noise_g_rms=0)
        v   = apply_signal_chain(sig)
        cnt = voltage_to_counts(v)
        recovered = counts_to_g(cnt)
        error = np.max(np.abs(recovered - sig))
        assert error < TOLERANCE_G, f"Round-trip error {error:.2f} g exceeds {TOLERANCE_G} g"

    def test_antialiasing_filter_attenuates_above_cutoff(self):
        """Signal at 20 kHz (above 4.8 kHz cutoff) must be attenuated by >20 dB"""
        _, sig_pass  = generate_vibration(freq_hz=100,   amplitude_g=50, noise_g_rms=0)
        _, sig_alias = generate_vibration(freq_hz=20000, amplitude_g=50, noise_g_rms=0)
        v_pass  = apply_signal_chain(sig_pass)
        v_alias = apply_signal_chain(sig_alias)
        rms_pass  = np.sqrt(np.mean((v_pass - np.mean(v_pass))**2))
        rms_alias = np.sqrt(np.mean((v_alias - np.mean(v_alias))**2))
        ratio_db = 20 * np.log10(rms_alias / (rms_pass + 1e-12))
        assert ratio_db < -20, f"AA filter attenuation only {-ratio_db:.1f} dB at 20 kHz"


### 2.4 - LM95172 SPI Tests
SPI word round-trip, 16-bit resolution, range validation, conversion-complete flag, geothermal sweep.

In [ ]:
%%writefile ics_sandbox/tests/test_lm95172.py
import pytest
from simulation.models.lm95172_mock import (
    LM95172Mock,
    CFG_RESOLUTION_13,
    CFG_RESOLUTION_16,
    TEMP_MIN_C,
    TEMP_MAX_C,
    ACCURACY_C,
)

TEMP_TOLERANCE_C = 0.1


class TestLM95172:

    def test_25c_round_trip(self):
        sensor = LM95172Mock()
        sensor.set_temperature(25.0)
        word = sensor.spi_read_temperature()
        result = sensor.parse_temperature(word)
        assert abs(result - 25.0) < TEMP_TOLERANCE_C

    def test_negative_temperature(self):
        sensor = LM95172Mock()
        sensor.set_temperature(-40.0)
        word = sensor.spi_read_temperature()
        result = sensor.parse_temperature(word)
        assert abs(result - (-40.0)) < TEMP_TOLERANCE_C

    def test_max_temperature(self):
        sensor = LM95172Mock()
        sensor.set_temperature(200.0)
        word = sensor.spi_read_temperature()
        result = sensor.parse_temperature(word)
        assert abs(result - 200.0) < TEMP_TOLERANCE_C

    def test_conversion_complete_flag(self):
        sensor = LM95172Mock()
        sensor.set_temperature(100.0)
        word = sensor.spi_read_temperature()
        assert word & 0x0004

    def test_out_of_range_raises(self):
        sensor = LM95172Mock()
        with pytest.raises(ValueError):
            sensor.set_temperature(201.0)
        with pytest.raises(ValueError):
            sensor.set_temperature(-41.0)

    def test_16bit_resolution_finer(self):
        s13 = LM95172Mock(CFG_RESOLUTION_13)
        s16 = LM95172Mock(CFG_RESOLUTION_16)
        s13.set_temperature(50.03125)
        s16.set_temperature(50.03125)
        r13 = s13.parse_temperature(s13.spi_read_temperature())
        r16 = s16.parse_temperature(s16.spi_read_temperature())
        assert abs(r16 - 50.03125) <= abs(r13 - 50.03125)

    def test_geothermal_operating_range(self):
        sensor = LM95172Mock()
        test_points = [-40, -20, 0, 25, 50, 75, 100, 125, 150, 175, 200]
        for t in test_points:
            sensor.set_temperature(float(t))
            word = sensor.spi_read_temperature()
            result = sensor.parse_temperature(word)
            assert abs(result - t) < ACCURACY_C, f"FAILED at {t}C: got {result:.4f}C"


### 2.5 - MAX31856 Tests
TC and CJ round-trip, fault detection, geothermal sweep, resolution accuracy.

In [ ]:
%%writefile ics_sandbox/tests/test_max31856.py
"""pytest tests for MAX31856 mock"""
import pytest
from simulation.models.max31856_mock import (
    MAX31856Mock, REG_CJTH, REG_LTCBH, REG_SR,
    FAULT_OPEN, FAULT_TC_HIGH, FAULT_CJ_HIGH,
    TC_RESOLUTION, CJ_RESOLUTION
)

TC_TOLERANCE_C = 0.05   # 5× resolution as round-trip tolerance
CJ_TOLERANCE_C = 0.03

class TestMAX31856:

    def test_tc_25c_round_trip(self):
        """Inject TC=25°C → parse LTCB bytes → recover ~25°C"""
        m = MAX31856Mock()
        m.set_tc_temperature(25.0)
        b = m.spi_read_register(REG_LTCBH)
        result = m.parse_tc_temperature(*b)
        assert abs(result - 25.0) < TC_TOLERANCE_C

    def test_tc_high_temperature(self):
        """Inject TC=1200°C (high geothermal) → round-trip"""
        m = MAX31856Mock()
        m.set_tc_temperature(1200.0)
        b = m.spi_read_register(REG_LTCBH)
        result = m.parse_tc_temperature(*b)
        assert abs(result - 1200.0) < TC_TOLERANCE_C

    def test_tc_negative_temperature(self):
        """Inject TC=−200°C → round-trip negative value"""
        m = MAX31856Mock()
        m.set_tc_temperature(-200.0)
        b = m.spi_read_register(REG_LTCBH)
        result = m.parse_tc_temperature(*b)
        assert abs(result - (-200.0)) < TC_TOLERANCE_C

    def test_cj_temperature_round_trip(self):
        """CJ at 85°C (hot electronics) → parse back correctly"""
        m = MAX31856Mock()
        m.set_cj_temperature(85.0)
        b = m.spi_read_register(REG_CJTH)
        result = m.parse_cj_temperature(*b)
        assert abs(result - 85.0) < CJ_TOLERANCE_C

    def test_cj_out_of_range_raises(self):
        """CJ temperature above 125°C must raise ValueError"""
        m = MAX31856Mock()
        with pytest.raises(ValueError):
            m.set_cj_temperature(130.0)

    def test_fault_open_circuit_detection(self):
        """Inject FAULT_OPEN bit → SR register must report it"""
        m = MAX31856Mock()
        m.inject_fault(FAULT_OPEN)
        sr = m.spi_read_register(REG_SR)[0]
        assert sr & FAULT_OPEN, "Open circuit fault not reported in SR"

    def test_fault_tc_high_detection(self):
        """Inject FAULT_TC_HIGH → SR register must report it"""
        m = MAX31856Mock()
        m.inject_fault(FAULT_TC_HIGH)
        sr = m.spi_read_register(REG_SR)[0]
        assert sr & FAULT_TC_HIGH

    def test_no_fault_clear_sr(self):
        """Normal operation → SR must be 0x00 (no faults)"""
        m = MAX31856Mock()
        sr = m.spi_read_register(REG_SR)[0]
        assert sr == 0x00

    def test_geothermal_temperature_sweep(self):
        """Sweep TC from 0 to 800°C in 50°C steps — all must round-trip within tolerance"""
        m = MAX31856Mock()
        for t in range(0, 850, 50):
            m.set_tc_temperature(float(t))
            b = m.spi_read_register(REG_LTCBH)
            result = m.parse_tc_temperature(*b)
            assert abs(result - t) < TC_TOLERANCE_C, \
                f"FAILED at {t}°C: got {result:.4f}°C"

    def test_resolution_accuracy(self):
        """Resolution must be 0.0078125°C — adjacent codes must differ by this amount"""
        m = MAX31856Mock()
        m.set_tc_temperature(100.0)
        b1 = m.spi_read_register(REG_LTCBH)
        r1 = m.parse_tc_temperature(*b1)
        m.set_tc_temperature(100.0 + TC_RESOLUTION)
        b2 = m.spi_read_register(REG_LTCBH)
        r2 = m.parse_tc_temperature(*b2)
        assert abs((r2 - r1) - TC_RESOLUTION) < 0.0001


---
## Section 3 - `generate_dashboard()` Function

Runs pytest against the sandbox, parses results, builds the HTML using the same template as the original `generate_dashboard.py`.


In [ ]:
def run_tests(sandbox):
    """Run pytest with --json-report against the sandbox."""
    sandbox    = Path(sandbox).resolve()
    json_path  = (sandbox / "reports" / "results.json").resolve()

    print("=" * 60)
    print("  ICS SENSOR VALIDATION - Running tests...")
    print("=" * 60)

    cmd = [
        sys.executable, "-m", "pytest", "tests/", "-v",
        "--json-report", f"--json-report-file={json_path}", "--tb=short",
    ]
    subprocess.run(cmd, cwd=sandbox)
    print()

    if not json_path.exists():
        raise RuntimeError("pytest did not produce results.json - "
                            "install pytest-json-report")
    return json_path


def parse_results(json_path):
    """Parse pytest --json-report output into per-test records and summary stats."""
    import datetime as _dt
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    tests    = data.get("tests", [])
    summary  = data.get("summary", {})
    env      = data.get("environment", {})
    created  = data.get("created", 0)
    duration = data.get("duration", 0)

    sensor_map = {
        "ADXL206":  "ADXL206HDZ",
        "ADXRS645": "ADXRS645HDYZ",
        "AT10TB":   "AT/10/TB",
        "LM95172":  "LM95172",
        "MAX31856": "MAX31856",
    }

    parsed = []
    for t in tests:
        nodeid   = t.get("nodeid", "")
        outcome  = t.get("outcome", "unknown").upper()
        dur      = t.get("duration", 0)

        parts    = nodeid.split("::")
        filepath = parts[0] if len(parts) > 0 else ""
        cls      = parts[1] if len(parts) > 1 else ""
        func     = parts[2] if len(parts) > 2 else parts[-1]

        fname  = Path(filepath).stem
        sensor = fname.replace("test_", "").upper()
        sensor_display = sensor_map.get(sensor, sensor)

        call    = t.get("call", {})
        longrepr = call.get("longrepr", "") if call and outcome == "FAILED" else ""

        parsed.append({
            "nodeid": nodeid, "file": filepath, "cls": cls, "func": func,
            "sensor": sensor_display, "outcome": outcome,
            "duration": round(dur * 1000, 1), "error": longrepr,
        })

    stats = {
        "total":    summary.get("total", len(tests)),
        "passed":   summary.get("passed", 0),
        "failed":   summary.get("failed", 0),
        "error":    summary.get("error", 0),
        "skipped":  summary.get("skipped", 0),
        "duration": round(duration, 2),
        "python":   env.get("Python", "unknown"),
        "pytest":   env.get("pytest", "unknown"),
        "platform": env.get("Platform", "unknown"),
        "timestamp": _dt.datetime.fromtimestamp(created).strftime("%d %b %Y  %H:%M:%S")
                     if created else _dt.datetime.now().strftime("%d %b %Y  %H:%M:%S"),
    }
    return parsed, stats


def group_by_sensor(tests):
    """Group test records by sensor for the dashboard."""
    groups = {}
    for t in tests:
        s = t["sensor"]
        if s not in groups:
            groups[s] = {"total": 0, "passed": 0, "failed": 0, "tests": []}
        groups[s]["total"] += 1
        groups[s]["tests"].append(t)
        if t["outcome"] == "PASSED":
            groups[s]["passed"] += 1
        else:
            groups[s]["failed"] += 1
    return groups


In [ ]:
# ─── STEP 4: BUILD HTML ───────────────────────────────────────────────────────
def build_html(tests, stats, groups):

    # Serialise test data for JS
    js_tests = json.dumps(tests, indent=2)
    js_stats = json.dumps(stats, indent=2)

    # Per-sensor summary rows
    sensor_rows = ""
    sensor_cards = ""
    sensor_color = {
        "ADXL206HDZ":  "#f59e0b",
        "ADXRS645HDYZ": "#14b8a6",
        "AT/10/TB":    "#3b82f6",
        "LM95172":     "#22c55e",
        "MAX31856":    "#ef4444",
    }
    for i, (sname, sg) in enumerate(groups.items()):
        rate    = round(sg["passed"] / sg["total"] * 100) if sg["total"] else 0
        color   = sensor_color.get(sname, "#8b92a5")
        status  = "PASS" if sg["failed"] == 0 else "FAIL"
        badge   = "badge-pass" if status == "PASS" else "badge-fail"
        selected = "selected" if i == 0 else ""
        key     = sname.lower().replace("/","").replace(" ","")

        sensor_cards += f"""
        <div class="sensor-card {selected}" id="sc-{key}" onclick="selectSensor('{key}')">
          <div class="sensor-name">{sname}</div>
          <div class="sensor-stats">
            <div class="stat-item"><div class="stat-val green">{sg['total']}</div><div class="stat-lbl">Tests</div></div>
            <div class="stat-item"><div class="stat-val {'green' if sg['failed']==0 else 'red'}">{sg['passed']}</div><div class="stat-lbl">Passed</div></div>
            <div class="stat-item"><div class="stat-val {'green' if sg['failed']==0 else 'red'}">{rate}%</div><div class="stat-lbl">Rate</div></div>
          </div>
        </div>"""

        sensor_rows += f"""
        <tr>
          <td><span style="font-family:var(--font-mono);font-size:12px;color:{color};">{sname}</span></td>
          <td style="font-family:var(--font-mono);font-size:13px;">{sg['total']}</td>
          <td style="color:var(--green);font-family:var(--font-mono);">{sg['passed']}</td>
          <td style="color:var(--red);font-family:var(--font-mono);">{sg['failed']}</td>
          <td style="font-family:var(--font-mono);">{rate}%</td>
          <td><span class="badge {badge}">{status}</span></td>
        </tr>"""

    pass_rate_overall = round(stats["passed"] / stats["total"] * 100) if stats["total"] else 0
    overall_color     = "var(--green)" if stats["failed"] == 0 else "var(--red)"

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>ICS Core — Sensor Validation Dashboard</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=Syne:wght@400;600;700;800&family=DM+Sans:wght@300;400;500&display=swap" rel="stylesheet">
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.0/chart.umd.min.js"></script>
<style>
:root {{
  --bg0:#0b0c0e;--bg1:#111318;--bg2:#181b22;--bg3:#1f232d;--bg4:#262b38;
  --amber:#f59e0b;--amber-dim:#92600a;--amber-glow:rgba(245,158,11,0.10);
  --teal:#14b8a6;--red:#ef4444;--green:#22c55e;--blue:#3b82f6;
  --text-primary:#e8eaf0;--text-secondary:#8b92a5;--text-muted:#4a5168;
  --border:rgba(255,255,255,0.07);--border-accent:rgba(245,158,11,0.3);
  --grid:rgba(255,255,255,0.022);
  --font-display:'Syne',sans-serif;--font-mono:'Space Mono',monospace;--font-body:'DM Sans',sans-serif;
}}
*,*::before,*::after{{box-sizing:border-box;margin:0;padding:0}}
html{{scroll-behavior:smooth}}
body{{background:var(--bg0);color:var(--text-primary);font-family:var(--font-body);font-size:15px;line-height:1.6;overflow-x:hidden}}
body::before{{content:'';position:fixed;inset:0;background-image:linear-gradient(var(--grid) 1px,transparent 1px),linear-gradient(90deg,var(--grid) 1px,transparent 1px);background-size:40px 40px;pointer-events:none;z-index:0}}

/* HEADER */
.header{{position:relative;z-index:10;border-bottom:1px solid var(--border);background:rgba(17,19,24,0.98);padding:0 40px}}
.header-top{{display:flex;align-items:center;justify-content:space-between;padding:18px 0;border-bottom:1px solid var(--border)}}
.logo-group{{display:flex;align-items:center;gap:14px}}
.logo-mark{{width:36px;height:36px;background:var(--amber);clip-path:polygon(50% 0%,100% 25%,100% 75%,50% 100%,0% 75%,0% 25%);display:flex;align-items:center;justify-content:center;flex-shrink:0}}
.logo-mark::after{{content:'';width:13px;height:13px;background:var(--bg0);clip-path:polygon(50% 0%,100% 25%,100% 75%,50% 100%,0% 75%,0% 25%)}}
.logo-text{{font-family:var(--font-display);font-size:13px;font-weight:700;letter-spacing:0.15em;text-transform:uppercase;color:var(--text-secondary)}}
.logo-text span{{color:var(--amber)}}
.header-meta{{display:flex;gap:12px;align-items:center;flex-wrap:wrap}}
.meta-pill{{font-family:var(--font-mono);font-size:11px;color:var(--text-muted);background:var(--bg3);border:1px solid var(--border);border-radius:4px;padding:4px 10px;letter-spacing:0.04em}}
.meta-pill.live{{border-color:var(--amber-dim);color:var(--amber);background:var(--amber-glow)}}
.meta-pill.live::before{{content:'● ';animation:pulse 2s ease-in-out infinite}}
@keyframes pulse{{0%,100%{{opacity:1}}50%{{opacity:0.25}}}}
.header-title{{padding:22px 0 18px}}
.page-title{{font-family:var(--font-display);font-size:30px;font-weight:800;letter-spacing:-0.02em;line-height:1.1}}
.page-title span{{color:var(--amber)}}
.page-subtitle{{font-size:13px;color:var(--text-secondary);margin-top:5px;font-weight:300}}

/* TABS */
.nav-tabs{{display:flex;border-top:1px solid var(--border)}}
.nav-tab{{font-family:var(--font-mono);font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:var(--text-muted);padding:13px 18px;cursor:pointer;border:none;background:none;border-bottom:2px solid transparent;transition:all 0.2s;position:relative;top:1px}}
.nav-tab:hover{{color:var(--text-secondary)}}
.nav-tab.active{{color:var(--amber);border-bottom-color:var(--amber)}}

/* LAYOUT */
.main{{position:relative;z-index:1;padding:28px 40px;max-width:1400px;margin:0 auto}}
.section{{display:none}}.section.active{{display:block}}
.two-col{{display:grid;grid-template-columns:1fr 1fr;gap:18px;margin-bottom:18px}}
.three-col{{display:grid;grid-template-columns:1fr 1fr 1fr;gap:18px;margin-bottom:18px}}
.full-col{{margin-bottom:18px}}

/* KPI */
.kpi-grid{{display:grid;grid-template-columns:repeat(5,1fr);gap:14px;margin-bottom:28px}}
.kpi-card{{background:var(--bg2);border:1px solid var(--border);border-radius:10px;padding:18px;position:relative;overflow:hidden}}
.kpi-card::before{{content:'';position:absolute;top:0;left:0;right:0;height:2px}}
.kpi-card.pass::before{{background:var(--green)}}.kpi-card.fail::before{{background:var(--red)}}
.kpi-card.warn::before{{background:var(--amber)}}.kpi-card.info::before{{background:var(--blue)}}
.kpi-card.total::before{{background:var(--text-muted)}}
.kpi-label{{font-family:var(--font-mono);font-size:10px;letter-spacing:0.12em;text-transform:uppercase;color:var(--text-muted);margin-bottom:8px}}
.kpi-value{{font-family:var(--font-display);font-size:34px;font-weight:800;line-height:1}}
.kpi-card.pass .kpi-value{{color:var(--green)}}.kpi-card.fail .kpi-value{{color:var(--red)}}
.kpi-card.warn .kpi-value{{color:var(--amber)}}.kpi-card.info .kpi-value{{color:var(--blue)}}
.kpi-card.total .kpi-value{{color:var(--text-primary)}}
.kpi-sub{{font-size:12px;color:var(--text-muted);margin-top:4px}}

/* PANEL */
.panel{{background:var(--bg2);border:1px solid var(--border);border-radius:10px;overflow:hidden}}
.panel-header{{display:flex;align-items:center;justify-content:space-between;padding:14px 18px;border-bottom:1px solid var(--border);background:var(--bg3)}}
.panel-title{{font-family:var(--font-display);font-size:13px;font-weight:700;color:var(--text-primary);display:flex;align-items:center;gap:8px}}
.dot{{width:7px;height:7px;border-radius:50%;background:var(--amber)}}
.panel-body{{padding:18px}}

/* TABLE */
.result-table{{width:100%;border-collapse:collapse}}
.result-table th{{font-family:var(--font-mono);font-size:10px;letter-spacing:0.1em;text-transform:uppercase;color:var(--text-muted);text-align:left;padding:10px 14px;border-bottom:1px solid var(--border);background:var(--bg3)}}
.result-table td{{padding:11px 14px;border-bottom:1px solid var(--border);font-size:13px;vertical-align:middle}}
.result-table tr:last-child td{{border-bottom:none}}
.result-table tr:hover td{{background:rgba(255,255,255,0.015)}}

/* BADGES */
.badge{{display:inline-flex;align-items:center;gap:5px;font-family:var(--font-mono);font-size:10px;font-weight:700;letter-spacing:0.08em;padding:3px 9px;border-radius:4px}}
.badge::before{{content:'●';font-size:8px}}
.badge-pass{{background:rgba(34,197,94,0.15);color:var(--green);border:1px solid rgba(34,197,94,0.3)}}
.badge-failed{{background:rgba(239,68,68,0.15);color:var(--red);border:1px solid rgba(239,68,68,0.3)}}
.badge-error{{background:rgba(239,68,68,0.15);color:var(--red);border:1px solid rgba(239,68,68,0.3)}}
.badge-skipped{{background:rgba(139,146,165,0.1);color:var(--text-muted);border:1px solid var(--border)}}

/* CHART */
.chart-wrap{{position:relative;height:260px}}.chart-wrap-lg{{position:relative;height:300px}}

/* SENSOR CARDS */
.sensor-grid{{display:grid;grid-template-columns:repeat(5,1fr);gap:14px;margin-bottom:18px}}
.sensor-card{{background:var(--bg2);border:1px solid var(--border);border-radius:10px;padding:16px;cursor:pointer;transition:all 0.2s}}
.sensor-card:hover{{border-color:var(--border-accent)}}
.sensor-card.selected{{border-color:var(--amber);background:linear-gradient(135deg,var(--bg3),var(--bg2));box-shadow:0 0 0 1px var(--amber-dim),inset 0 0 40px var(--amber-glow)}}
.sensor-name{{font-family:var(--font-display);font-size:13px;font-weight:700;color:var(--text-primary);margin-bottom:10px}}
.sensor-stats{{display:flex;gap:12px}}
.stat-item{{flex:1}}
.stat-val{{font-family:var(--font-display);font-size:20px;font-weight:700;line-height:1}}
.stat-val.green{{color:var(--green)}}.stat-val.red{{color:var(--red)}}
.stat-lbl{{font-size:10px;color:var(--text-muted);font-family:var(--font-mono);text-transform:uppercase;letter-spacing:0.06em;margin-top:2px}}

/* CONSOLE */
.console{{background:var(--bg0);border:1px solid var(--border);border-radius:8px;padding:14px;font-family:var(--font-mono);font-size:12px;line-height:1.8;max-height:420px;overflow-y:auto}}
.cl{{display:flex;gap:10px}}
.ct{{color:var(--text-muted);flex-shrink:0;min-width:80px}}
.cp{{color:var(--green)}}.cf{{color:var(--red)}}.ci{{color:var(--teal)}}.cw{{color:var(--amber)}}.cd{{color:var(--text-muted)}}

/* ERROR BOX */
.error-box{{background:rgba(239,68,68,0.06);border:1px solid rgba(239,68,68,0.2);border-radius:6px;padding:10px 14px;margin-top:8px;font-family:var(--font-mono);font-size:11px;color:var(--red);line-height:1.6;white-space:pre-wrap;word-break:break-word}}

/* SCROLLBAR */
::-webkit-scrollbar{{width:5px;height:5px}}
::-webkit-scrollbar-track{{background:var(--bg1)}}
::-webkit-scrollbar-thumb{{background:var(--bg4);border-radius:3px}}

/* FOOTER */
.footer{{position:relative;z-index:1;border-top:1px solid var(--border);padding:18px 40px;display:flex;align-items:center;justify-content:space-between}}
.footer-text{{font-family:var(--font-mono);font-size:11px;color:var(--text-muted);letter-spacing:0.04em}}

@keyframes fadeIn{{from{{opacity:0;transform:translateY(6px)}}to{{opacity:1;transform:translateY(0)}}}}
.fade-in{{animation:fadeIn 0.35s ease forwards}}
@media(max-width:1100px){{.kpi-grid{{grid-template-columns:repeat(3,1fr)}}.sensor-grid{{grid-template-columns:repeat(2,1fr)}}.two-col,.three-col{{grid-template-columns:1fr}}}}
</style>
</head>
<body>

<header class="header">
  <div class="header-top">
    <div class="logo-group">
      <div class="logo-mark"></div>
      <div>
        <div class="logo-text"><span>ICS</span> Core System</div>
        <div style="font-size:11px;color:var(--text-muted);font-family:var(--font-mono);margin-top:2px">GEO-TVP-001 · Rev A · ZerdaLabs</div>
      </div>
    </div>
    <div class="header-meta">
      <div class="meta-pill">dsPIC33AK256MC205-H/M7</div>
      <div class="meta-pill" id="pill-python">Python {stats['python']}</div>
      <div class="meta-pill" id="pill-pytest">pytest {stats['pytest']}</div>
      <div class="meta-pill" id="pill-time">{stats['timestamp']}</div>
      <div class="meta-pill live">Phase A — Simulation</div>
    </div>
  </div>
  <div class="header-title">
    <h1 class="page-title">Sensor Validation <span>Dashboard</span></h1>
    <p class="page-subtitle">Geothermal Downhole Sensor Package · Pre-prototype simulation validation · Generated from live pytest output</p>
  </div>
  <nav class="nav-tabs">
    <button class="nav-tab active" onclick="showSection('overview',this)">Overview</button>
    <button class="nav-tab" onclick="showSection('sensors',this)">Sensor Analysis</button>
    <button class="nav-tab" onclick="showSection('tests',this)">All Tests</button>
    <button class="nav-tab" onclick="showSection('console',this)">Console</button>
  </nav>
</header>

<main class="main">

<!-- OVERVIEW -->
<div class="section active fade-in" id="section-overview">
  <div class="kpi-grid">
    <div class="kpi-card total"><div class="kpi-label">Total Tests</div><div class="kpi-value" id="k-total">{stats['total']}</div><div class="kpi-sub">across all sensors</div></div>
    <div class="kpi-card pass"><div class="kpi-label">Passed</div><div class="kpi-value" id="k-passed">{stats['passed']}</div><div class="kpi-sub" id="k-passrate">{pass_rate_overall}% pass rate</div></div>
    <div class="kpi-card fail"><div class="kpi-label">Failed</div><div class="kpi-value" id="k-failed">{stats['failed']}</div><div class="kpi-sub">regressions</div></div>
    <div class="kpi-card warn"><div class="kpi-label">Skipped</div><div class="kpi-value" id="k-skipped">{stats['skipped']}</div><div class="kpi-sub">not run</div></div>
    <div class="kpi-card info"><div class="kpi-label">Duration</div><div class="kpi-value" style="font-size:26px;margin-top:4px">{stats['duration']}s</div><div class="kpi-sub">total runtime</div></div>
  </div>

  <div class="two-col">
    <div class="panel">
      <div class="panel-header"><div class="panel-title"><div class="dot"></div>Pass rate by sensor</div></div>
      <div class="panel-body"><div class="chart-wrap"><canvas id="barChart"></canvas></div></div>
    </div>
    <div class="panel">
      <div class="panel-header"><div class="panel-title"><div class="dot"></div>Test distribution</div></div>
      <div class="panel-body"><div class="chart-wrap"><canvas id="doughnutChart"></canvas></div></div>
    </div>
  </div>

  <div class="panel full-col">
    <div class="panel-header">
      <div class="panel-title"><div class="dot"></div>Per-sensor summary</div>
      <span style="font-family:var(--font-mono);font-size:10px;color:var(--text-muted)">LIVE DATA FROM PYTEST</span>
    </div>
    <div style="padding:0">
      <table class="result-table">
        <thead><tr><th>Sensor</th><th>Total</th><th>Passed</th><th>Failed</th><th>Pass Rate</th><th>Status</th></tr></thead>
        <tbody>{sensor_rows}</tbody>
      </table>
    </div>
  </div>
</div>

<!-- SENSOR ANALYSIS -->
<div class="section fade-in" id="section-sensors">
  <div class="sensor-grid">{sensor_cards}</div>
  <div class="panel full-col">
    <div class="panel-header">
      <div class="panel-title"><div class="dot"></div><span id="sensor-panel-title">Select a sensor above</span></div>
    </div>
    <div class="panel-body" style="padding:0">
      <table class="result-table" id="sensor-detail-table">
        <thead><tr><th>Test Function</th><th>Duration (ms)</th><th>Status</th><th>Error</th></tr></thead>
        <tbody id="sensor-detail-body"></tbody>
      </table>
    </div>
  </div>
  <div class="panel full-col" style="margin-top:18px">
    <div class="panel-header"><div class="panel-title"><div class="dot"></div>Duration profile (ms)</div></div>
    <div class="panel-body"><div class="chart-wrap-lg"><canvas id="durationChart"></canvas></div></div>
  </div>
</div>

<!-- ALL TESTS -->
<div class="section fade-in" id="section-tests">
  <div class="panel full-col">
    <div class="panel-header">
      <div class="panel-title"><div class="dot"></div>All test results</div>
      <span style="font-family:var(--font-mono);font-size:10px;color:{overall_color}" id="summary-badge">{stats['passed']}/{stats['total']} PASSED</span>
    </div>
    <div style="padding:0">
      <table class="result-table">
        <thead><tr><th>Sensor</th><th>Class</th><th>Test Function</th><th>Duration (ms)</th><th>Status</th></tr></thead>
        <tbody id="all-tests-body"></tbody>
      </table>
    </div>
  </div>
</div>

<!-- CONSOLE -->
<div class="section fade-in" id="section-console">
  <div class="panel full-col">
    <div class="panel-header">
      <div class="panel-title"><div class="dot"></div>Test console output</div>
      <span style="font-family:var(--font-mono);font-size:10px;color:var(--text-muted)">pytest tests/ -v --json-report</span>
    </div>
    <div class="panel-body">
      <div class="console" id="console-out"></div>
    </div>
  </div>
  <div class="panel full-col" style="margin-top:18px">
    <div class="panel-header"><div class="panel-title"><div class="dot"></div>How to refresh this dashboard</div></div>
    <div class="panel-body">
      <div class="console">
        <div class="cl"><span class="ct">#</span><span class="cd">From geo_sensor_validation/ folder</span></div>
        <div class="cl"><span class="ct">$</span><span class="ci">python generate_dashboard.py</span></div>
        <div class="cl"><span class="ct">&nbsp;</span><span class="cd">&nbsp;</span></div>
        <div class="cl"><span class="ct">#</span><span class="cd">Or run just the tests first then generate</span></div>
        <div class="cl"><span class="ct">$</span><span class="ci">pytest tests/ -v --json-report --json-report-file=reports/results.json</span></div>
        <div class="cl"><span class="ct">$</span><span class="ci">python generate_dashboard.py</span></div>
        <div class="cl"><span class="ct">&nbsp;</span><span class="cd">&nbsp;</span></div>
        <div class="cl"><span class="ct">#</span><span class="cw">Phase B — hardware mode</span></div>
        <div class="cl"><span class="ct">$</span><span class="cw">set SENSOR_MODE=hardware && python generate_dashboard.py</span></div>
      </div>
    </div>
  </div>
</div>

</main>

<footer class="footer">
  <div class="footer-text">GEO-TVP-001 · Rev A · Internal Use Only · ZerdaLabs / University of Bath</div>
  <div class="footer-text" id="footer-ts">Generated: {stats['timestamp']}</div>
</footer>

<script>
const TESTS  = {js_tests};
const STATS  = {js_stats};

const SENSOR_COLORS = {{
  "ADXL206HDZ":  "#f59e0b",
  "ADXRS645HDYZ":"#14b8a6",
  "AT/10/TB":    "#3b82f6",
  "LM95172":     "#22c55e",
  "MAX31856":    "#ef4444",
}};

// Group tests by sensor
function groupBySensor(tests) {{
  const g = {{}};
  tests.forEach(t => {{
    if (!g[t.sensor]) g[t.sensor] = [];
    g[t.sensor].push(t);
  }});
  return g;
}}

const GROUPS = groupBySensor(TESTS);
const SENSOR_KEYS = Object.keys(GROUPS);

// ── CHART DEFAULTS ──
const CD = {{
  responsive:true, maintainAspectRatio:false,
  plugins:{{
    legend:{{labels:{{color:'#8b92a5',font:{{family:"'Space Mono',monospace",size:10}},boxWidth:10}}}},
    tooltip:{{backgroundColor:'#1f232d',borderColor:'#262b38',borderWidth:1,titleColor:'#e8eaf0',bodyColor:'#8b92a5',titleFont:{{family:"'Space Mono',monospace",size:11}},bodyFont:{{family:"'DM Sans',sans-serif",size:12}}}}
  }},
  scales:{{
    x:{{ticks:{{color:'#4a5168',font:{{family:"'Space Mono',monospace",size:10}}}},grid:{{color:'rgba(255,255,255,0.03)'}},border:{{color:'rgba(255,255,255,0.07)'}}}},
    y:{{ticks:{{color:'#4a5168',font:{{family:"'Space Mono',monospace",size:10}}}},grid:{{color:'rgba(255,255,255,0.03)'}},border:{{color:'rgba(255,255,255,0.07)'}}}}
  }}
}};

// ── NAV ──
function showSection(id, btn) {{
  document.querySelectorAll('.section').forEach(s => s.classList.remove('active'));
  document.querySelectorAll('.nav-tab').forEach(t => t.classList.remove('active'));
  document.getElementById('section-' + id).classList.add('active');
  btn.classList.add('active');
  if (id === 'sensors') {{
    initSensorSection();
    selectSensorByKey(Object.keys(GROUPS)[0]);
  }}
}}

// ── BADGE HTML ──
function badge(outcome) {{
  const cls = 'badge-' + outcome.toLowerCase();
  return `<span class="badge ${{cls}}">${{outcome}}</span>`;
}}

// ── ALL TESTS TABLE ──
function buildAllTests() {{
  const tbody = document.getElementById('all-tests-body');
  TESTS.forEach(t => {{
    const color = SENSOR_COLORS[t.sensor] || '#8b92a5';
    const tr = document.createElement('tr');
    tr.innerHTML = `
      <td><span style="font-family:var(--font-mono);font-size:12px;color:${{color}}">${{t.sensor}}</span></td>
      <td style="font-family:var(--font-mono);font-size:11px;color:var(--text-muted)">${{t.cls}}</td>
      <td style="font-family:var(--font-mono);font-size:12px;color:var(--teal)">${{t.func}}</td>
      <td style="font-family:var(--font-mono);font-size:12px">${{t.duration}}</td>
      <td>${{badge(t.outcome)}}</td>
    `;
    if (t.outcome !== 'PASSED' && t.error) {{
      const errRow = document.createElement('tr');
      errRow.innerHTML = `<td colspan="5"><div class="error-box">${{t.error.substring(0,400)}}</div></td>`;
      tbody.appendChild(tr);
      tbody.appendChild(errRow);
    }} else {{
      tbody.appendChild(tr);
    }}
  }});
}}

// ── OVERVIEW CHARTS ──
function initOverviewCharts() {{
  const sensors = SENSOR_KEYS;
  const passed  = sensors.map(s => GROUPS[s].filter(t => t.outcome === 'PASSED').length);
  const failed  = sensors.map(s => GROUPS[s].filter(t => t.outcome !== 'PASSED').length);
  const totals  = sensors.map(s => GROUPS[s].length);

  new Chart(document.getElementById('barChart'), {{
    type:'bar',
    data:{{
      labels: sensors,
      datasets:[
        {{label:'Passed',data:passed,backgroundColor:'rgba(34,197,94,0.7)',borderColor:'#22c55e',borderWidth:1,borderRadius:4}},
        {{label:'Failed',data:failed,backgroundColor:'rgba(239,68,68,0.5)',borderColor:'#ef4444',borderWidth:1,borderRadius:4}}
      ]
    }},
    options:{{...CD}}
  }});

  new Chart(document.getElementById('doughnutChart'), {{
    type:'doughnut',
    data:{{
      labels: sensors.map((s,i) => s+' ('+totals[i]+')'),
      datasets:[{{
        data: totals,
        backgroundColor: sensors.map(s => SENSOR_COLORS[s] || '#8b92a5'),
        borderColor:'#111318', borderWidth:3, hoverOffset:6
      }}]
    }},
    options:{{responsive:true,maintainAspectRatio:false,plugins:{{
      legend:{{position:'right',labels:{{color:'#8b92a5',font:{{family:"'Space Mono',monospace",size:10}},boxWidth:10,padding:12}}}},
      tooltip:CD.plugins.tooltip
    }}}}
  }});
}}

// ── SENSOR SECTION ──
let durationChart = null;

function initSensorSection() {{
  // Wire up sensor card clicks
  SENSOR_KEYS.forEach(s => {{
    const key = s.toLowerCase().replace(/\\//g,'').replace(/ /g,'');
    const el = document.getElementById('sc-' + key);
    if (el) el.onclick = () => selectSensorByKey(s);
  }});
}}

function selectSensorByKey(sensorName) {{
  // Highlight card
  SENSOR_KEYS.forEach(s => {{
    const key = s.toLowerCase().replace(/\\//g,'').replace(/ /g,'');
    const el = document.getElementById('sc-' + key);
    if (el) el.classList.toggle('selected', s === sensorName);
  }});

  const tests = GROUPS[sensorName] || [];
  document.getElementById('sensor-panel-title').textContent = sensorName + ' — test details';

  // Detail table
  const tbody = document.getElementById('sensor-detail-body');
  tbody.innerHTML = '';
  tests.forEach(t => {{
    const tr = document.createElement('tr');
    tr.innerHTML = `
      <td style="font-family:var(--font-mono);font-size:12px;color:var(--teal)">${{t.func}}</td>
      <td style="font-family:var(--font-mono);font-size:12px">${{t.duration}}</td>
      <td>${{badge(t.outcome)}}</td>
      <td style="font-size:12px;color:var(--red);font-family:var(--font-mono)">${{t.outcome !== 'PASSED' ? (t.error||'').substring(0,80) : '—'}}</td>
    `;
    tbody.appendChild(tr);
  }});

  // Duration bar chart
  if (durationChart) durationChart.destroy();
  const color = SENSOR_COLORS[sensorName] || '#f59e0b';
  durationChart = new Chart(document.getElementById('durationChart'), {{
    type: 'bar',
    data: {{
      labels: tests.map(t => t.func.replace('test_','')),
      datasets: [{{
        label: 'Duration (ms)',
        data: tests.map(t => t.duration),
        backgroundColor: tests.map(t => t.outcome === 'PASSED' ? color + 'bb' : 'rgba(239,68,68,0.6)'),
        borderColor: tests.map(t => t.outcome === 'PASSED' ? color : '#ef4444'),
        borderWidth: 1, borderRadius: 4
      }}]
    }},
    options: {{
      ...CD,
      indexAxis: 'y',
      scales: {{
        x: {{ ...CD.scales.x, title:{{display:true,text:'ms',color:'#4a5168',font:{{size:10}}}} }},
        y: {{ ...CD.scales.y, ticks:{{color:'#8b92a5',font:{{family:"'Space Mono',monospace",size:10}}}} }}
      }}
    }}
  }});
}}

// ── CONSOLE ──
function buildConsole() {{
  const div = document.getElementById('console-out');
  const platform = STATS.platform || 'unknown';
  const python   = STATS.python   || 'unknown';
  const pytest   = STATS.pytest   || 'unknown';

  const lines = [
    {{c:'ci', t:'==================== test session starts ===================='}},
    {{c:'cd', t:'platform ' + platform + ' -- Python ' + python + ', pytest-' + pytest}},
    {{c:'cd', t:'rootdir: geo_sensor_validation/'}},
    {{c:'cd', t:'collected ' + STATS.total + ' items'}},
    {{c:'cd', t:' '}},
  ];

  TESTS.forEach(t => {{
    const status = t.outcome === 'PASSED' ? 'PASSED' : 'FAILED';
    const cls = t.outcome === 'PASSED' ? 'cp' : 'cf';
    lines.push({{c:cls, t: t.file + '::' + t.cls + '::' + t.func + ' ' + status}});
    if (t.outcome !== 'PASSED' && t.error) {{
      t.error.split('\\n').slice(0,6).forEach(l => lines.push({{c:'cf', t:'  ' + l}}));
    }}
  }});

  const total   = STATS.total;
  const passed  = STATS.passed;
  const failed  = STATS.failed;
  const dur     = STATS.duration;
  const summary = failed === 0
    ? `==================== ${{passed}} passed in ${{dur}}s ====================`
    : `==================== ${{passed}} passed, ${{failed}} failed in ${{dur}}s ====================`;
  lines.push({{c:'cd', t:' '}});
  lines.push({{c: failed === 0 ? 'cp' : 'cf', t: summary}});

  lines.forEach(l => {{
    const row = document.createElement('div');
    row.className = 'cl';
    row.innerHTML = `<span class="ct"></span><span class="${{l.c}}">${{l.t}}</span>`;
    div.appendChild(row);
  }});
}}

// ── INIT ──
window.addEventListener('DOMContentLoaded', () => {{
  buildAllTests();
  buildConsole();
  initOverviewCharts();
}});
</script>
</body>
</html>"""

    return html


In [ ]:
def write_html(html, output_path):
    """Write the HTML string to disk."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(html, encoding="utf-8")
    return output_path


def generate_dashboard(sandbox=None, html_output=None):
    """End-to-end: run pytest in the sandbox, build HTML, return its file path.

    Args:
        sandbox    : sandbox project root. Default: SANDBOX from Section 0.
        html_output: path for the HTML. Default: <sandbox>/reports/sensor_dashboard.html

    Returns:
        (html_path, stats) - Path to the HTML file, and the summary stats dict.
    """
    sandbox = Path(sandbox or SANDBOX).resolve()

    json_path    = run_tests(sandbox)
    tests, stats = parse_results(json_path)
    groups       = group_by_sensor(tests)
    html         = build_html(tests, stats, groups)

    if html_output is None:
        html_output = sandbox / "reports" / "sensor_dashboard.html"
    html_path = write_html(html, html_output)

    print("=" * 60)
    print(f"  Dashboard written to: {html_path}")
    print(f"  Tests:   {stats['total']}  |  Passed: {stats['passed']}  |  Failed: {stats['failed']}")
    print(f"  Runtime: {stats['duration']}s")
    print("=" * 60)
    return html_path, stats


---
## Section 4 - Run Dashboard

Returns the HTML file path. No image is generated.


In [ ]:
html_path, stats = generate_dashboard()

print(f"\nHTML file path : {html_path}")
print(f"File size      : {html_path.stat().st_size / 1024:.1f} kB")

html_path


In [ ]:
import sys, subprocess
result = subprocess.run([sys.executable, "-m", "pip", "show", "pytest-json-report"],
                         capture_output=True, text=True)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
print("RC:", result.returncode)